# PlayNow: qué jugar según el momento

Este cuaderno muestra cómo elegir un juego de la biblioteca según el tiempo disponible, la fatiga estimada y lo que la persona quiere jugar. Carga los datos, prepara el motor y compara tres momentos de juego para la misma persona.

Los títulos son reales. Las características de los juegos son estimaciones de diseño; los usuarios, bibliotecas, sesiones y comentarios son simulados. Las conexiones con Steam, PlayStation, Xbox, la agenda y el reloj se representan mediante archivos CSV.

La columna llamada `gsr` se calcula desde la variabilidad cardíaca (HRV); no contiene una lectura independiente de conductancia de la piel. Los comentarios también se construyen desde las fichas de los juegos y sirven para mostrar cómo se podrían extraer características de textos.

Se usa la semilla 42 para repetir los cálculos aleatorios.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Se localiza la raíz tanto si se abre el cuaderno desde el proyecto como desde notebooks.
ROOT = Path.cwd().resolve()
if not (ROOT / "playnow").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from playnow.engine import (
    SEED,
    INTENTION_TARGETS,
    PlayNowEngine,
    fatigue_from_wearable,
    lexicon_scores,
)

# Los ajustes de pandas ayudan a leer las tablas sin recortar tanto las descripciones.
DATA = ROOT / "data"
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 120)
print("semilla", SEED)

semilla 42


## Datos del catálogo y de las bibliotecas

El catálogo reúne las características de cada videojuego: cuánto tarda en enganchar, qué esfuerzo exige y qué tipo de experiencia ofrece. La biblioteca indica qué juegos tiene cada persona, cuántas horas les ha dedicado y cuánto lleva sin abrirlos.

Las dos tablas se cruzan por `game_id`, el identificador del juego. Así, las recomendaciones se limitan a la biblioteca de la persona elegida.

In [2]:
# Cada CSV guarda un conjunto distinto: juegos, comentarios, personas, bibliotecas e historial.
catalog = pd.read_csv(DATA / "catalogo_videojuegos_profundo.csv")
comments = pd.read_csv(DATA / "comentarios_foros_videojuegos.csv")
users = pd.read_csv(DATA / "usuarios_perfiles_ocupacionales.csv")
library = pd.read_csv(DATA / "biblioteca_usuario.csv")
sessions = pd.read_csv(DATA / "sesiones_contextuales_wearable.csv")

# Se revisa el tamaño de las tablas y se muestran algunas fichas del catálogo.
print(f"juegos={len(catalog)} comentarios={len(comments)} usuarios={len(users)} filas_biblioteca={len(library)} sesiones={len(sessions)}")
print("perfiles", users["perfil_ocupacional"].value_counts().to_dict())
print("eventos de cierre", sessions["evento_cierre"].value_counts().to_dict())
catalog[["titulo", "genero", "es_indie", "minutos_hasta_diversion", "duracion_minima_sesion_min", "carga_cognitiva", "valencia", "arousal", "cuadrante", "mu_relajacion", "mu_historia", "score_post_trabajo"]].head(8)

juegos=68 comentarios=136 usuarios=15 filas_biblioteca=109 sesiones=58
perfiles {'alta_carga': 6, 'fin_de_semana': 5, 'competitivo': 4}
eventos de cierre {'GUARDADO_FIN_NATURAL': 44, 'SALIDA_PREMATURA_FRUSTRACION': 8, 'CAMBIO_JUEGO': 6}


,titulo,genero,es_indie,minutos_hasta_diversion,duracion_minima_sesion_min,carga_cognitiva,valencia,arousal,cuadrante,mu_relajacion,mu_historia,score_post_trabajo
0,Hades,roguelike,1,3,20,0.45,0.35,0.75,HVHA,0.10,0.25,0.500
1,Stardew Valley,simulacion,1,5,20,0.25,0.70,-0.55,HVLA,0.90,0.20,0.917
2,Elden Ring,accion-rpg,0,15,45,0.70,-0.20,0.55,LVHA,0.05,0.35,0.500
3,Baldur's Gate 3,rpg,0,25,75,0.80,0.40,0.15,HVHA,0.15,0.95,0.175
4,Rocket League,deportes,0,1,15,0.30,0.25,0.85,HVHA,0.05,0.00,0.500
5,Cyberpunk 2077,rpg,0,20,50,0.55,0.10,0.35,HVHA,0.10,0.85,0.175
6,Celeste,plataformas,1,2,20,0.40,0.15,0.40,HVHA,0.15,0.40,0.500
7,Slay the Spire,roguelike,1,4,25,0.65,0.20,-0.10,HVLA,0.35,0.10,0.500


## Características obtenidas de los comentarios

El léxico es un listado de frases que se buscan en los comentarios. Expresiones como «después del trabajo», «cuesta retomar» o «frustrante» ayudan a puntuar si un juego sirve para desconectar, si es fácil volver a él y cuánto confort o frustración sugiere el texto.

Cada juego tiene dos comentarios simulados, redactados a partir de sus propias características. Sus puntajes se promedian y se incorporan al vector del juego. Este ejemplo muestra el recorrido del texto al número; no aporta opiniones independientes de jugadores.

In [3]:
# Se toma un comentario por juego y se muestran tres ejemplos con sus puntajes.
sample = comments.groupby("game_id", as_index=False).first().head(3)
scored = sample.copy()
scored["lexico"] = scored["texto"].map(lambda text: lexicon_scores(text))
for row in scored.itertuples(index=False):
    print(row.game_id)
    print(row.texto)
    print(row.lexico)
    print("---")

# Estas comprobaciones conservan visible el origen simulado de los datos.
assert comments["fuente"].eq("corpus_sintetico").all()
assert catalog["fuente_atributos"].eq("estimacion_de_diseno_no_medicion").all()

amongus
Se puede abrir un rato, aunque no es la opción más suave. Retomar fácil, controles simples, se vuelve a entrar y no olvidas los controles. Se siente justo y se puede dejar sin castigo.
{'score_post_trabajo': 0.5, 'score_facil_retomar': 0.9, 'score_juego_confort': 1.0, 'riesgo_frustracion_lexico': 0.0}
---
animal
Ideal después del trabajo, partida corta para desconectar cuando llego cansado, sin pensar mucho. Retomar fácil, controles simples, se vuelve a entrar y no olvidas los controles. Se siente justo y se puede dejar sin castigo.
{'score_post_trabajo': 0.917, 'score_facil_retomar': 0.9, 'score_juego_confort': 1.0, 'riesgo_frustracion_lexico': 0.0}
---
baba
Se puede abrir un rato, aunque no es la opción más suave. Retomar fácil, controles simples, se vuelve a entrar y no olvidas los controles. Se siente justo y se puede dejar sin castigo.
{'score_post_trabajo': 0.5, 'score_facil_retomar': 0.9, 'score_juego_confort': 1.0, 'riesgo_frustracion_lexico': 0.0}
---


## Familias de juegos y plano de Russell

Primero se ponen las características en una escala comparable. Después, PCA reduce la cantidad de dimensiones conservando al menos el 95 % de la variación del catálogo. K-Means usa esa representación para agrupar juegos parecidos.

La curva del codo permite elegir cuántos grupos usar. Su inercia mide qué tan dispersos quedan los juegos respecto al centro de cada grupo. El punto elegido es el más alejado de la recta que une los extremos de la curva.

El plano de Russell muestra dos características: **valencia**, qué tan agradable se estima la experiencia, y **activación** (`arousal`), qué tan tranquila o intensa resulta. Los colores representan las familias de juegos. Estos grupos describen el catálogo; no determinan el orden de recomendación.

In [4]:
# Al preparar el motor también se calculan las familias de juegos y la curva del codo.
engine = PlayNowEngine(catalog, library, users, sessions)
print("k elegido", int(engine.cluster_model.n_clusters))
print(engine.elbow.to_string(index=False))

# El primer gráfico muestra cómo cambia la dispersión al aumentar la cantidad de grupos.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(engine.elbow["k"], engine.elbow["inertia"], marker="o")
axes[0].axvline(engine.cluster_model.n_clusters, color="tab:red", linestyle="--")
axes[0].set_title("Método del codo")
axes[0].set_xlabel("k")
axes[0].set_ylabel("inercia")

# El segundo sitúa cada juego por valencia y activación, con un color por familia.
plane = engine.catalog
# Se muestran algunos títulos de cada familia para interpretar los grupos.
for cluster_id, group in plane.groupby("cluster_id"):
    axes[1].scatter(group["valencia"], group["arousal"], s=28, label=f"familia {cluster_id}")
axes[1].axhline(0, color="black", linewidth=0.6)
axes[1].axvline(0, color="black", linewidth=0.6)
axes[1].set_xlim(-1.05, 1.05)
axes[1].set_ylim(-1.05, 1.05)
axes[1].set_xlabel("valencia")
axes[1].set_ylabel("activación")
axes[1].set_title("Plano de Russell")
axes[1].legend(fontsize=8, loc="best")
fig.tight_layout()
plt.show()

plane.groupby("cluster_id")["titulo"].apply(lambda titles: ", ".join(titles.head(4))).to_frame("ejemplos")

k elegido 5
 k    inertia
 2 599.826601
 3 432.432218
 4 337.725417
 5 268.196689
 6 225.768693
 7 193.951504
 8 176.984079
 9 157.616835
10 137.017698


/var/folders/y6/l3g48_9j1gnb2sn86zgly0g40000gp/T/ipykernel_9855/1819188096.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,ejemplos
cluster_id,
0,"Elden Ring, Rocket League, Celeste, Hollow Knight"
1,"Slay the Spire, Civilization VI, Portal 2, Baba Is You"
2,"Stardew Valley, Forza Horizon 5, Unpacking, A Short Hike"
3,"Hades, Vampire Survivors, God of War, Among Us"
4,"Baldur's Gate 3, Cyberpunk 2077, Disco Elysium, Red Dead Redemption 2"


## Carga mental admisible para la sesión

La fatiga se estima con pulso, variabilidad cardíaca (HRV) y horas de sueño. Solo el pulso se compara con la frecuencia habitual de Andrés; HRV y sueño usan referencias fijas. Es una regla de diseño para simular el contexto.

El modelo difuso recibe esa fatiga y los minutos libres. Sus categorías se solapan: por ejemplo, un tiempo puede pertenecer parcialmente a «corto» y a «medio». Las reglas combinan esos grados para representar decisiones como «si hay mucha fatiga o poco tiempo, conviene una carga baja».

La salida se resume mediante el centroide, que toma el centro de equilibrio de la forma obtenida. El resultado se lleva a una escala de 0 a 1 para compararlo con la carga cognitiva de cada juego.

In [5]:
# Se usa el pulso habitual de Andrés como referencia para las tres sesiones.
andres_baseline = float(users.loc[users["user_id"] == "U01", "hr_baseline"].iloc[0])
# Cada escenario contiene descripción, minutos, pulso, HRV, sueño e intención.
contexts = [
    ("Martes 21:30, relajarme", 35, 92, 28, 4.5, "relajarme"),
    ("Sábado 16:00, historia", 180, 66, 52, 8.0, "historia"),
    ("Viernes 20:00, reto", 50, 78, 40, 6.5, "reto"),
]
rows = []
for label, minutes, hr, hrv, sleep, intention in contexts:
    # Primero se estima la fatiga; después las reglas difusas calculan la carga admisible.
    fatigue = fatigue_from_wearable(hr, hrv, sleep, andres_baseline)
    load = engine.fuzzy.admissible_load(minutes, fatigue)
    rows.append({
        "contexto": label,
        "minutos": minutes,
        "fatiga": round(fatigue, 1),
        "carga_admisible": round(load, 3),
        "intencion": INTENTION_TARGETS[intention]["label"],
    })
pd.DataFrame(rows)

,contexto,minutos,fatiga,carga_admisible,intencion
0,"Martes 21:30, relajarme",35,78.1,0.160,Relajarme / Desconectar
1,"Sábado 16:00, historia",180,5.6,0.844,Sumergirme en una historia
2,"Viernes 20:00, reto",50,36.6,0.410,Reto exigente / Competitivo


## Cómo se calcula el orden de recomendación

Se usa la biblioteca de Andrés (`U01`) como conjunto de candidatos. La intención elegida define un perfil de experiencia que se compara con las características de cada juego.

La similitud base combina el coseno de los vectores, con peso 0,55, y la cercanía de Russell, con peso 0,45. Como valencia y activación están entre -1 y 1, la mayor distancia entre dos puntos es la diagonal del plano, $2\sqrt{2}$:

$$
\mathrm{Russell}=\operatorname{clip}\left(1-\frac{d}{2\sqrt{2}},\,0,\,1\right)
$$

Después se aplica el esfuerzo de retomar el juego. Una penalización positiva resta; una negativa suma un bono cuando hay al menos dos horas y poca fatiga. El resultado se limita siempre al intervalo [0, 1], también cuando hay bono:

$$
\mathrm{Similitud}=\operatorname{clip}\left(\mathrm{SimilitudBase}-\mathrm{PenalizacionReenganche},\,0,\,1\right)
$$

Así, un bono puede mejorar la afinidad sin hacer que una coincidencia supere el máximo de la escala.

El puntaje combina esa similitud, el ajuste de carga y la señal de otras personas del mismo perfil ocupacional:

$$
\mathrm{Puntaje}(u,i,c)=\Phi(i,c)\cdot\big(\alpha\,\mathrm{Similitud}(i,c)+\beta\,\mathrm{AjusteCarga}(i,c)+\gamma\,\mathrm{Vecinos}(u,i)\big)
$$

`u` identifica a la persona, `i` al juego y `c` al contexto. La viabilidad $\Phi$ reduce el puntaje si el juego encaja peor con el tiempo o la carga de la sesión. El ajuste de carga aumenta cuanto más cerca está la exigencia del juego de la carga admisible.

Los pesos son decisiones de diseño. Si los vecinos aportan puntajes para al menos dos juegos, se usan 0,55 para similitud, 0,30 para ajuste de carga y 0,15 para vecinos. En caso contrario se usan 0,65, 0,35 y 0. La señal de vecinos solo se incorpora cuando la adecuación del juego a la intención es al menos 0,45.

Tras ordenar por puntaje, el motor puede dar el tercer puesto a un juego casi sin usar si su viabilidad, similitud y puntaje cumplen los umbrales previstos. `puesto_base` guarda el orden anterior a ese ajuste.

In [6]:
def show(ranking, k=3):
    """Muestra las primeras recomendaciones con sus componentes de puntaje."""
    columns = ["puesto", "titulo", "score", "horas_jugadas", "phi_viabilidad", "similitud", "ajuste_difuso", "score_colaborativo", "penalizacion_reenganche", "explicacion"]
    return ranking[columns].head(k)

# La lista por horas sirve como referencia y no depende de la intención de la sesión.
baseline = engine.baseline("U01")
print("Línea base por horas (igual en los tres contextos)")
print(baseline.head(5).to_string(index=False))

# Se guardan los identificadores de la biblioteca para comprobar cada recomendación.
owned_ids = set(library.loc[library["user_id"] == "U01", "game_id"])
rankings = {}
for label, minutes, hr, hrv, sleep, intention in contexts:
    fatigue = fatigue_from_wearable(hr, hrv, sleep, andres_baseline)
    ranking = engine.rank("U01", minutes, fatigue, intention)
    # Ningún juego recomendado debe quedar fuera de la biblioteca de Andrés.
    assert set(ranking["game_id"]).issubset(owned_ids)
    # Las dos similitudes deben mantenerse en [0, 1], incluso con bono de reenganche.
    assert ranking["similitud_russell"].between(0.0, 1.0).all()
    assert ranking["similitud"].between(0.0, 1.0).all()
    rankings[intention] = ranking
    print("\n==", label, "==")
    print(show(ranking).drop(columns=["explicacion"]).to_string(index=False))
    print(ranking.iloc[0]["explicacion"])
    print("métricas", engine.list_metrics(ranking))

Línea base por horas (igual en los tres contextos)
 puesto game_id          titulo  horas_jugadas  dias_sin_jugar  es_indie
      1   hades           Hades            140               2         1
      2    civ6 Civilization VI             48              15         0
      3 stardew  Stardew Valley             36               1         1
      4  rocket   Rocket League             32               5         0
      5   elden      Elden Ring             28              24         0

== Martes 21:30, relajarme ==
 puesto         titulo  score  horas_jugadas  phi_viabilidad  similitud  ajuste_difuso  score_colaborativo  penalizacion_reenganche
      1      Unpacking 0.9458              0          1.0000     0.9092         0.9997               0.972                   0.0000
      2 Stardew Valley 0.8390             36          0.9238     0.9830         0.9103               0.630                   0.0041
      3    Coffee Talk 0.8307              6          1.0000     0.9706         0.98

## Qué cambia al usar la experiencia de otras personas

Se repite la sesión del martes con la señal de vecinos activada y desactivada. En ambos casos se mantiene el cálculo de contenido, viabilidad, carga y reenganche; al desactivar los vecinos también se redistribuyen sus pesos. Además, se desactiva en ambos casos la promoción de juegos poco jugados para separar ese efecto.

La comparación permite observar si las sesiones de otras personas del mismo perfil cambian el primer puesto. Los historiales son pequeños y simulados: el ejemplo muestra cómo funciona esa señal.

In [7]:
minutes, hr, hrv, sleep = 35, 92, 28, 4.5
fatigue = fatigue_from_wearable(hr, hrv, sleep, andres_baseline)
# Se compara la misma sesión con y sin vecinos, sin promover juegos poco jugados.
with_neighbors = engine.rank("U01", minutes, fatigue, "relajarme", apply_backlog_slot=False)
content_only = engine.rank("U01", minutes, fatigue, "relajarme", apply_backlog_slot=False, use_collaborative=False)
print("Con vecinos", with_neighbors["titulo"].head(3).tolist())
print("Solo contenido", content_only["titulo"].head(3).tolist())
# Se comprueba que la señal de vecinos cambia el primer puesto de este ejemplo.
assert with_neighbors.iloc[0]["titulo"] != content_only.iloc[0]["titulo"]

# La tabla reúne la referencia por horas y los tres escenarios de la misma biblioteca.
comparison = pd.DataFrame({
    "horas": baseline["titulo"].head(3).tolist(),
    "relajarme": rankings["relajarme"]["titulo"].head(3).tolist(),
    "historia": rankings["historia"]["titulo"].head(3).tolist(),
    "reto": rankings["reto"]["titulo"].head(3).tolist(),
})
comparison

Con vecinos ['Unpacking', 'Stardew Valley', 'Coffee Talk']
Solo contenido ['Coffee Talk', 'Unpacking', 'Stardew Valley']


,horas,relajarme,historia,reto
0,Hades,Unpacking,Baldur's Gate 3,Rocket League
1,Civilization VI,Stardew Valley,Disco Elysium,Hades
2,Stardew Valley,Coffee Talk,The Witcher 3,Celeste


## Qué describen los indicadores

Se calculan tres medidas sobre los tres primeros juegos de cada lista:

- **Diversidad:** distancia media entre los juegos en valencia, activación, carga cognitiva y densidad narrativa.
- **Juegos poco jugados:** fracción de la lista con menos de dos horas acumuladas.
- **Presencia indie:** vale 1 si aparece al menos un juego indie y 0 si no aparece ninguno.

Estas medidas permiten describir y comparar las listas de cada escenario. No miden satisfacción real de jugadores.

In [8]:
# Se reúnen los indicadores y los tres primeros títulos para comparar los escenarios.
summary = []
for intention, ranking in rankings.items():
    metrics = engine.list_metrics(ranking)
    metrics["intencion"] = intention
    metrics["top"] = ", ".join(ranking["titulo"].head(3))
    summary.append(metrics)
pd.DataFrame(summary)[["intencion", "top", "diversidad_intra_lista", "cobertura_backlog", "exposicion_indie"]]

,intencion,top,diversidad_intra_lista,cobertura_backlog,exposicion_indie
0,relajarme,"Unpacking, Stardew Valley, Coffee Talk",0.4400,0.3333,1.0
1,historia,"Baldur's Gate 3, Disco Elysium, The Witcher 3",0.4723,0.0000,1.0
2,reto,"Rocket League, Hades, Celeste",0.4669,0.0000,1.0
